# quantkit — Colab runner

Thin runner only: all logic lives in the `quantkit` package. Runtime → Change runtime type → **L4**.
Run the setup cells, then the section for the phase you're on.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,clocks.max.sm,power.limit --format=csv
import torch; assert 'L4' in torch.cuda.get_device_name(0), 'Not an L4 — switch runtime before collecting numbers'

In [ ]:
# Checkpoints and results persist on Drive; the HF model cache stays on local disk
# (reading a freshly written multi-GB safetensors back through the Drive mount corrupted it).
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/quantkit'
os.makedirs(WORK, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'
# Llama-3.2 is gated: accept the license on its HF page, and store your token in
# Colab Secrets (key icon) as HF_TOKEN. Scripts run as subprocesses and inherit this.
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
REPO = 'https://github.com/ankitsharma285/quantkit.git'
!rm -rf /content/quantkit && git clone -q $REPO /content/quantkit
%cd /content/quantkit
!pip install -q -e '.[dev]'

In [ ]:
!python -m pytest -q

In [ ]:
import json, glob

def show(results_dir):
    rows = [json.load(open(f)) for f in sorted(glob.glob(f'{results_dir}/*.json'))]
    print(f"{'run':34s} {'wikitext2':>10s} {'c4':>10s} {'quant s':>8s}")
    for r in rows:
        p = r['perplexity']
        w = p.get('wikitext2', {}).get('ppl', float('nan'))
        c = p.get('c4', {}).get('ppl', float('nan'))
        t = r.get('quant_seconds', float('nan'))
        print(f"{r['label']:34s} {w:10.3f} {c:10.3f} {t:8.1f}")

## Phase 1 — harness + RTN (done)

In [ ]:
R = f'{WORK}/results/day1'
!python scripts/quantize_eval.py --method fp16 --results-dir $R
!python scripts/quantize_eval.py --method rtn --group-size 128 --save $WORK/ckpt/rtn-g128 --results-dir $R
!python scripts/quantize_eval.py --method rtn --group-size -1 --results-dir $R
!python scripts/quantize_eval.py --method rtn --bits 3 --group-size 128 --fake --results-dir $R

In [ ]:
# Reload the saved int4 checkpoint: ppl must match the rtn-w4-g128 run exactly.
!python scripts/quantize_eval.py --load $WORK/ckpt/rtn-g128 --datasets wikitext2 --results-dir $R

In [ ]:
show(f'{WORK}/results/day1')

## Phase 2 — GPTQ

**Smoke test first** (16 calibration samples, 8 eval windows, ~1–2 min): catches wiring problems before the full runs.

In [ ]:
!python scripts/quantize_eval.py --method gptq --nsamples 16 --datasets wikitext2 --max-windows 8 \
    --results-dir $WORK/results/sanity

**Main runs.** Same calibration set (C4, 128 × 2048, seed 0) for every run. Expected order on WikiText-2: fp16 9.75 < GPTQ < RTN 11.73.

In [ ]:
R2 = f'{WORK}/results/phase2'
os.makedirs(R2, exist_ok=True)
# Bring the Phase 1 baselines along so the Phase 2 table is self-contained.
!cp "{WORK}"/results/day1/*_fp16.json "{WORK}"/results/day1/*_rtn-w4-g128.json "{R2}"/
!python scripts/quantize_eval.py --method gptq --save $WORK/ckpt/gptq-g128 --results-dir $R2

In [ ]:
# The packed GPTQ checkpoint must reproduce the in-memory perplexity exactly.
!python scripts/quantize_eval.py --load $WORK/ckpt/gptq-g128 --datasets wikitext2 --results-dir $R2

In [ ]:
# Variants: act-order (static groups), 3-bit, and GPTQModel's default damping (0.05 vs the paper's 0.01).
!python scripts/quantize_eval.py --method gptq --act-order --results-dir $R2
!python scripts/quantize_eval.py --method gptq --bits 3 --fake --results-dir $R2
!python scripts/quantize_eval.py --method gptq --damp 0.05 --results-dir $R2

In [ ]:
show(R2)

### Diagnosis: GPTQ without act-order worse than RTN

First run (Nov 19): GPTQ W4 g128 gave 34.2 (reloaded checkpoint, damp 0.01) and 81.6 (the damp 0.05 run, which
overwrote the main JSON because damping wasn't in the label; fixed). Act-order gave 10.78, better than RTN's 11.73.

Hypothesis: the first token of each calibration window is an attention sink with massive activations in a
few hidden channels, and it dominates the Hessians. Test: rerun with those positions left out of the
Hessians, and check how lopsided each Hessian diagonal is.

In [ ]:
R3 = f'{WORK}/results/phase2_diag'
os.makedirs(R3, exist_ok=True)
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --results-dir $R3
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --skip-first-tokens 1 --results-dir $R3
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --skip-first-tokens 4 --results-dir $R3
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --act-order --skip-first-tokens 1 --results-dir $R3
show(R3)

In [ ]:
# Which modules blow up, and how lopsided are their Hessians?
!python scripts/inspect_gptq.py {R3}/*.json

### Diagnosis, round 2: mechanism and noise

Round 1 (WikiText-2): plain GPTQ 34.17; leaving the first token out of the Hessians 11.16 (first 4: 11.10);
act-order 10.78; act-order + skip first token 11.24; RTN 11.73. Vanilla GPTQModel also breaks (50.86).

Open questions:
1. *Mechanism.* Does fixing group scales from the original weights (static groups, no reordering) also
   rescue plain GPTQ? If yes, the failure is dynamic group scales being chosen after error feedback has
   inflated the weights (`scale growth` in the inspect output shows it directly). If no, reordering is what helps.
2. *Noise.* Is act-order's 10.78 vs 11.24 real or calibration-sample luck? Two more calibration seeds.

In [ ]:
R4 = f'{WORK}/results/phase2_diag2'
os.makedirs(R4, exist_ok=True)
run = 'python scripts/quantize_eval.py --method gptq --datasets wikitext2 --results-dir ' + R4
!{run}                                   # rerun plain GPTQ to record scale growth
!{run} --static-groups
!{run} --static-groups --skip-first-tokens 1
!{run} --act-order --calib-seed 1
!{run} --act-order --calib-seed 2
show(R4)

In [ ]:
!python scripts/inspect_gptq.py {R4}/*.json

### Reference: GPTQModel

Run in a **fresh session** (Runtime → Disconnect and delete runtime, then the setup cells): installing GPTQModel
changes packages.

Round 1: vanilla settings (no act-order) gave 50.86 vs quantkit's 34.17. Both are in the broken regime, where
small implementation differences get amplified, so that pair can't test parity. The parity check uses the
configuration that works: act-order with static groups.

In [ ]:
!pip install -q gptqmodel
# GPTQModel imports transformers' AutoProcessor, which imports torchaudio; Colab's torchaudio
# is built for a different CUDA than its torch. quantkit doesn't use audio, so remove it.
!pip uninstall -y -q torchaudio

In [ ]:
R2 = f'{WORK}/results/phase2'
# Matched, healthy comparison: act-order with static groups in both implementations.
# Done when this is within 1% of quantkit's gptq-w4-g128-ao (10.780).
!python scripts/reference_gptqmodel.py --act-order --out /content/ckpt/ref-gptqmodel-ao --results-dir $R2
show(R2)

### Parity gap: where do quantkit and GPTQModel diverge?

Matched act-order run: GPTQModel 11.086 vs quantkit 10.780 on WikiText-2 (2.8%; 6.0% on C4), outside the 1% target.
Compare the two checkpoints module by module, in execution order. Same session as the reference run
(GPTQModel installed). Each step skips itself if its checkpoint already exists.

In [ ]:
import os
if not os.path.exists('/content/ckpt/ref-gptqmodel-ao'):
    !python scripts/reference_gptqmodel.py --act-order --out /content/ckpt/ref-gptqmodel-ao --results-dir /content/tmp_results
if not os.path.exists('/content/ckpt/gptq-ao'):
    !python scripts/quantize_eval.py --method gptq --act-order --datasets wikitext2 --max-windows 4 --save /content/ckpt/gptq-ao --results-dir /content/tmp_results
!python scripts/compare_checkpoints.py /content/ckpt/gptq-ao /content/ckpt/ref-gptqmodel-ao | tee $WORK/results/phase2/compare_ao.txt

### Closing Phase 2: is the parity gap real or numerical chaos?

The checkpoint comparison: group scales identical in every module; ~9% of codes already differ in layer 0's
first module (identical weights and inputs); each checkpoint is equally far from the fp16 weights; no growing
drift with depth. GPTQ feeds every rounding error into all later columns, so one flipped rounding early in a row
changes the rest of that row. Two tests decide whether the 0.31 perplexity gap is a real difference:

1. GPTQModel on calibration seeds 1 and 2: compare 3-seed means, not single runs.
2. quantkit with TF32 matmuls (a tiny numerical change): how many codes flip vs quantkit's own fp32 run, and
   how much perplexity moves.

Same session as the GPTQModel install. About 25 minutes.

In [ ]:
R2 = f'{WORK}/results/phase2'
!python scripts/reference_gptqmodel.py --act-order --calib-seed 1 --datasets wikitext2 --out /content/ckpt/ref-ao-s1 --results-dir $R2
!python scripts/reference_gptqmodel.py --act-order --calib-seed 2 --datasets wikitext2 --out /content/ckpt/ref-ao-s2 --results-dir $R2
R4 = f'{WORK}/results/phase2_diag2'
!python scripts/quantize_eval.py --method gptq --act-order --tf32 --datasets wikitext2 --save /content/ckpt/gptq-ao-tf32 --results-dir $R4
show(R2); show(R4)

In [ ]:
if not os.path.exists('/content/ckpt/gptq-ao'):
    !python scripts/quantize_eval.py --method gptq --act-order --datasets wikitext2 --max-windows 4 --save /content/ckpt/gptq-ao --results-dir /content/tmp_results
!python scripts/compare_checkpoints.py /content/ckpt/gptq-ao /content/ckpt/gptq-ao-tf32 | tee $WORK/results/phase2/compare_ao_vs_tf32.txt | head -12

### Last Phase 2 test: is the gap bfloat16 vs float16?

GPTQModel is worse on every seed (11.01 ± 0.02 vs 10.71 ± 0.11). Numerical noise is ruled out: TF32 flips as many codes as
the GPTQModel comparison but moves perplexity by 0.1%. GPTQModel loads Llama-3.2 in bfloat16 for both calibration and
evaluation; quantkit uses float16. Three runs separate the two effects (one GPTQModel session, about 15 minutes):

| Run | Isolates |
|---|---|
| quantkit fp16 model evaluated in bf16 | pure evaluation-dtype effect, no quantization |
| quantkit GPTQ act-order, calibrated and evaluated in bf16 | quantkit under GPTQModel's dtype |
| GPTQModel act-order checkpoint evaluated in float16 | GPTQModel under quantkit's evaluation dtype |

If the bf16 quantkit run lands near 11.0 and the float16-evaluated GPTQModel run near 10.7–10.8, dtype explains the gap
and Phase 2 closes.

In [ ]:
R5 = f'{WORK}/results/phase2_dtype'
os.makedirs(R5, exist_ok=True)
!python scripts/quantize_eval.py --method fp16 --dtype bfloat16 --datasets wikitext2 --results-dir $R5
!python scripts/quantize_eval.py --method gptq --act-order --dtype bfloat16 --fake --datasets wikitext2 --results-dir $R5
!python scripts/reference_gptqmodel.py --act-order --reuse --eval-dtype float16 --datasets wikitext2 --out /content/ckpt/ref-gptqmodel-ao --results-dir $R5
show(R5)

## Phase 4 — Triton W4A16 kernel

Plain session (no GPTQModel needed). Run the setup cells first.

**1. Correctness on the GPU.** The CPU test suite already ran the kernels in Triton's interpreter; this adds the
autotuned kernels on the real Llama-3.2-1B layer shapes, M from 1 to 255. The first run compiles and autotunes
(a few minutes).

In [ ]:
!python -m pytest -q -m gpu tests/test_kernel.py

**2. End-to-end correctness.** The saved RTN W4 g128 checkpoint evaluated through the kernel must reproduce the
reference-path perplexity (11.728) up to summation order.

In [ ]:
R6 = f'{WORK}/results/phase4'
os.makedirs(R6, exist_ok=True)
!python scripts/quantize_eval.py --load $WORK/ckpt/rtn-g128 --backend triton --datasets wikitext2 --results-dir $R6

**3. Micro-benchmark.** All four layer shapes, M = 1…256, against fp16 cuBLAS on identical weights. About 15–20
minutes, most of it autotuning on first use. The table prints as it goes; the JSON (timings per round, achieved
bandwidth, chosen configs) is saved to Drive. Expect at M = 1: fp16 around 300 GB/s worth of weights per call, the
kernel ideally several times faster; at large M, fp16 cuBLAS should win.

In [ ]:
!python scripts/bench_kernel.py --out $WORK/results/phase4/bench_l4.json

## Phase 5 — kernel tuning

Phase 4: correct and faster than fp16 at decode, but 11–40% of the L4's spec bandwidth (cuBLAS: 36–60%).
Three candidate fixes, each a switch so its effect is measured separately:
- **unpack once:** load each packed int32 once per tile instead of once per nibble;
- **GEMM with BLOCK_M = 16:** decode in one launch with a direct fp16 store (split-K needs zero-fill + atomics + cast);
- **GEMV kernel for M = 1:** no tensor cores, no 16-row padding;

plus **measured dispatch**: on first use for a shape, time the candidate kernels and keep the fastest.

Plain session; run the setup cells first. **1. Practical bandwidth ceiling** (1 minute).

In [ ]:
R7 = f'{WORK}/results/phase5'
os.makedirs(R7, exist_ok=True)
!python scripts/bw_ceiling.py --out $R7/bw_ceiling.json

**2. Profile with Nsight Compute** (about 5 minutes): gate/up_proj at M = 1 with cuBLAS, the Phase 4 split-K kernel
(per-nibble loads), and the new dispatcher; then k/v_proj, which sat at a ~16 µs floor. Full reports go to Drive;
the cell prints the key lines.

In [ ]:
import subprocess
NCU = ('ncu --profile-from-start off --clock-control none --section SpeedOfLight --section MemoryWorkloadAnalysis '
       '--section Occupancy --section LaunchStats --section WarpStateStats ')
runs = {
    'gateup_fp16':          '--shape gate/up_proj --M 1 --impl fp16',
    'gateup_splitk_nibble': '--shape gate/up_proj --M 1 --impl splitk --unpack per_nibble',
    'gateup_auto_once':     '--shape gate/up_proj --M 1 --impl auto',
    'kv_fp16':              '--shape k/v_proj --M 1 --impl fp16',
    'kv_auto_once':         '--shape k/v_proj --M 1 --impl auto',
}
keys = ('auto dispatch', 'Duration', 'DRAM Throughput', 'Memory Throughput', 'Compute (SM) Throughput',
        'Achieved Occupancy', 'Registers Per Thread', 'Grid Size', 'Block Size')
for name, a in runs.items():
    out = subprocess.run(NCU + 'python scripts/profile_kernel.py ' + a, shell=True, capture_output=True, text=True).stdout
    open(f'{R7}/ncu_{name}.txt', 'w').write(out)
    print(f'===== {name}')
    for line in out.splitlines():
        s = line.strip()
        if s.startswith(keys) or ('(' in s and ')x(' in s):
            print('  ' + s[:110])

**3. Benchmark** every path × unpack variant, all shapes, M = 1…256 (about 25–30 minutes, mostly autotuning).

In [ ]:
!python scripts/bench_kernel.py --ceiling-json $R7/bw_ceiling.json --out $R7/bench_l4_v2.json

**4. Correctness with the new defaults**: GPU tests (now every path × unpack variant), and whole-model perplexity
through the kernel (must still be 11.728).

In [ ]:
!python -m pytest -q -m gpu tests/test_kernel.py
!python scripts/quantize_eval.py --load $WORK/ckpt/rtn-g128 --backend triton --datasets wikitext2 --results-dir $R7

## Phase 6 — end-to-end decode

Plain session; setup cells first. **1. Save the GPTQ act-order checkpoint** (the model the headline uses; about 6 minutes).
WikiText-2 should reproduce 10.780.

In [ ]:
R8 = f'{WORK}/results/phase6'
os.makedirs(R8, exist_ok=True)
!python scripts/quantize_eval.py --method gptq --act-order --save $WORK/ckpt/gptq-ao-g128 --datasets wikitext2 --results-dir $R8

**2. Correctness of the fast path**: the saved checkpoint through the Triton kernels with q/k/v and gate/up fused must
match the reference path (10.780) up to summation order.

In [ ]:
!python scripts/quantize_eval.py --load $WORK/ckpt/gptq-ao-g128 --backend triton --fuse --datasets wikitext2 --results-dir $R8

**3. Decode benchmark**: batch 1, 128-token prompt, 256 generated tokens, greedy. Four configs, about 15 minutes
(the first W4 config autotunes new shapes).

Bandwidth-bound ceilings at the measured ~232–243 GB/s: fp16 reads ~2.47 GB per token (≈ 95 tok/s); W4A16 reads ~0.51 GB
of int4 linears plus the 0.53 GB fp16 lm_head (≈ 225 tok/s), so ~2.4× is the best this model can show without
quantizing lm_head.

In [ ]:
CK = f'{WORK}/ckpt/gptq-ao-g128'
!python scripts/bench_decode.py --configs fp16 fp16+fuse w4:$CK w4+fuse:$CK --out $R8/decode_l4.json

### Phase 6b — quantized lm_head and downstream accuracy

Plain session; setup cells first.

**1. What does an int4 lm_head cost in accuracy?** (about 10 minutes) lm_head is RTN-quantized and untied from the
fp16 embedding. Three runs: the fp16 model with only lm_head quantized (isolates its cost), the GPTQ act-order model
plus lm_head at group size 128, and at 64. Compare with fp16 9.751 / 14.012 and GPTQ act-order 10.780 / 15.320.

In [ ]:
R9 = f'{WORK}/results/phase6_lmh'
os.makedirs(R9, exist_ok=True)
CK = f'{WORK}/ckpt/gptq-ao-g128'
!python scripts/quantize_eval.py --method fp16 --quantize-lm-head --backend triton --results-dir $R9
!python scripts/quantize_eval.py --load $CK --quantize-lm-head --backend triton --results-dir $R9
!python scripts/quantize_eval.py --load $CK --quantize-lm-head --lm-head-group-size 64 --backend triton --results-dir $R9
show(R9)

**2. Decode speed with an int4 lm_head** (about 10 minutes): fp16 again for a same-session reference.

In [ ]:
!python scripts/bench_decode.py --configs fp16 w4+fuse:$CK w4+lmh+fuse:$CK --out $R9/decode_l4_lmh.json

**3. Downstream accuracy** with lm-evaluation-harness, zero-shot: ARC-easy, ARC-challenge, HellaSwag, PIQA,
WinoGrande. First a 50-example smoke test (a few minutes), then the full run: four models, roughly 10–15 minutes each
(estimate). Results merge into one JSON, so the configs can be split across sessions if Colab disconnects.

In [ ]:
!pip install -q 'lm-eval>=0.4'
!python scripts/eval_tasks.py --configs fp16 w4:$CK --limit 50 --out /content/tasks_smoke.json

In [ ]:
RTN = f'{WORK}/ckpt/rtn-g128'
!python scripts/eval_tasks.py --configs fp16 w4:$RTN w4:$CK w4+lmh:$CK --out $R9/tasks_l4.json